# 30 — First-Order Greeks: A Survey

Compute every first-order Greek for every major asset class using `jax.grad`.

| Greek | Definition | `argnums` in BSM |
|-------|-----------|------------------|
| Delta | ∂V/∂S | 0 |
| Vega  | ∂V/∂σ | 5 |
| Theta | -∂V/∂T | 2 |
| Rho   | ∂V/∂r | 3 |
| Psi   | ∂V/∂q | 4 |

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from ql_jax.engines.analytic.black_scholes_merton import bsm_price
from ql_jax.engines.analytic.heston import heston_price
from ql_jax.instruments.swap import swap_npv
from ql_jax.engines.analytic.black_cap_floor import black_cap_price

---
## 1. Equity: BSM Greeks

In [ ]:
S, K, T, r, q, sigma = 100.0, 100.0, 1.0, 0.03, 0.01, 0.2

# All first-order Greeks in one call via value_and_grad
price = bsm_price(S, K, T, r, q, sigma, 1.0)
delta = jax.grad(bsm_price, argnums=0)(S, K, T, r, q, sigma, 1.0)
vega  = jax.grad(bsm_price, argnums=5)(S, K, T, r, q, sigma, 1.0)
theta = -jax.grad(bsm_price, argnums=2)(S, K, T, r, q, sigma, 1.0)
rho   = jax.grad(bsm_price, argnums=3)(S, K, T, r, q, sigma, 1.0)
psi   = jax.grad(bsm_price, argnums=4)(S, K, T, r, q, sigma, 1.0)

print(f"BSM ATM Call (S={S}, K={K}, T={T}, σ={sigma})")
print(f"  Price : {float(price):.6f}")
print(f"  Delta : {float(delta):.6f}")
print(f"  Vega  : {float(vega):.6f}")
print(f"  Theta : {float(theta):.6f}")
print(f"  Rho   : {float(rho):.6f}")
print(f"  Psi   : {float(psi):.6f}")

---
## 2. jacfwd: All Greeks in One Forward Pass

In [ ]:
def bsm_wrapper(params):
    return bsm_price(params[0], params[1], params[2], params[3], params[4], params[5], 1.0)

params = jnp.array([S, K, T, r, q, sigma])
jac = jax.jacfwd(bsm_wrapper)(params)

labels = ['∂V/∂S (Δ)', '∂V/∂K', '∂V/∂T (-Θ)', '∂V/∂r (ρ)', '∂V/∂q (ψ)', '∂V/∂σ (ν)']
for lbl, val in zip(labels, jac):
    print(f"  {lbl:15s} = {float(val):+.6f}")

---
## 3. Heston Model Greeks

In [ ]:
v0, kappa, theta_h, xi, rho_h = 0.04, 1.5, 0.04, 0.3, -0.7

p = heston_price(S, K, T, r, q, v0, kappa, theta_h, xi, rho_h, 1)

# Spot delta
delta_h = jax.grad(heston_price, argnums=0)(S, K, T, r, q, v0, kappa, theta_h, xi, rho_h, 1)
# Vol-of-vol vega
d_xi = jax.grad(heston_price, argnums=8)(S, K, T, r, q, v0, kappa, theta_h, xi, rho_h, 1)
# Correlation sensitivity
d_rho = jax.grad(heston_price, argnums=9)(S, K, T, r, q, v0, kappa, theta_h, xi, rho_h, 1)
# v0 sensitivity
d_v0 = jax.grad(heston_price, argnums=5)(S, K, T, r, q, v0, kappa, theta_h, xi, rho_h, 1)

print(f"Heston ATM Call")
print(f"  Price     : {float(p):.6f}")
print(f"  Delta     : {float(delta_h):.6f}")
print(f"  ∂V/∂v₀    : {float(d_v0):.6f}")
print(f"  ∂V/∂ξ     : {float(d_xi):.6f}")
print(f"  ∂V/∂ρ     : {float(d_rho):.6f}")

---
## 4. Greek Surfaces

In [ ]:
strikes = jnp.linspace(70, 130, 60)
maturities = jnp.linspace(0.1, 2.0, 50)
KK, TT = jnp.meshgrid(strikes, maturities)

delta_fn = jax.jit(jax.vmap(jax.vmap(
    lambda K, T: jax.grad(bsm_price, argnums=0)(S, K, T, r, q, sigma, 1.0),
    in_axes=(0, 0)), in_axes=(0, 0)))

vega_fn = jax.jit(jax.vmap(jax.vmap(
    lambda K, T: jax.grad(bsm_price, argnums=5)(S, K, T, r, q, sigma, 1.0),
    in_axes=(0, 0)), in_axes=(0, 0)))

D = delta_fn(KK, TT)
V = vega_fn(KK, TT)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

c1 = ax1.contourf(np.array(KK), np.array(TT), np.array(D), levels=20, cmap='RdYlGn')
fig.colorbar(c1, ax=ax1)
ax1.set_xlabel('Strike')
ax1.set_ylabel('Maturity')
ax1.set_title('Delta Surface')

c2 = ax2.contourf(np.array(KK), np.array(TT), np.array(V), levels=20, cmap='viridis')
fig.colorbar(c2, ax=ax2)
ax2.set_xlabel('Strike')
ax2.set_ylabel('Maturity')
ax2.set_title('Vega Surface')

fig.suptitle('BSM Greek Surfaces via AD', fontsize=14)
fig.tight_layout()
plt.show()

---
## 5. AD vs Finite Differences: Accuracy

In [ ]:
# Compare AD delta to central FD at various bump sizes
ad_delta = float(jax.grad(bsm_price, argnums=0)(S, K, T, r, q, sigma, 1.0))

bumps = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10]
fd_errors = []

for h in bumps:
    fd = (bsm_price(S + h, K, T, r, q, sigma, 1.0) - bsm_price(S - h, K, T, r, q, sigma, 1.0)) / (2 * h)
    fd_errors.append(abs(float(fd) - ad_delta))

plt.figure(figsize=(8, 5))
plt.loglog(bumps, fd_errors, 'o-', color='coral', label='|FD - AD|')
plt.axhline(y=1e-15, color='green', linestyle='--', label='Machine epsilon')
plt.xlabel('Bump Size h')
plt.ylabel('Absolute Error')
plt.title('Finite Difference Error vs Bump Size (AD is exact)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"AD delta (exact): {ad_delta:.15f}")
print(f"Best FD error   : {min(fd_errors):.2e} at h={bumps[fd_errors.index(min(fd_errors))]}")

---
## 6. Exercises

1. **Cap Greeks**: Compute Delta, Vega, Rho for a cap using AD.
2. **Forward mode**: Use `jax.jvp` to compute Delta and compare speed with `jax.grad`.
3. **Exotic Greeks**: Compute AD Greeks for a barrier option near the barrier.

---
*End of Notebook 30*